# Complete Betting System (EXTRA)

**Chapter 9: From Predictions to Profit: Navigating the World of Soccer Betting**

## Learning Objectives

- Build an end-to-end betting system
- Integrate predictions, odds, and bet sizing
- Implement comprehensive backtesting
- Track multiple performance metrics
- Handle real-world considerations
- Build a complete betting dashboard
- Understand practical betting challenges

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

np.random.seed(42)

print("Complete Betting System Ready!")

## Introduction

This notebook brings together **everything** from Chapter 9:

1. ✅ Odds analysis and value identification
2. ✅ Prediction model integration
3. ✅ Kelly Criterion bet sizing
4. ✅ Comprehensive backtesting
5. ✅ Performance tracking and reporting
6. ✅ Risk management

We'll build a **production-ready betting system** from scratch!

---

# 1. Complete Betting System Class

Let's build a comprehensive betting system.

In [ ]:
class SoccerBettingSystem:
    """
    Complete end-to-end soccer betting system.
    """
    
    def __init__(self, initial_bankroll=1000, kelly_fraction=0.5, min_edge=0.03):
        """
        Initialize betting system.
        
        Args:
            initial_bankroll: Starting capital
            kelly_fraction: Fraction of Kelly to use (0.5 = half Kelly)
            min_edge: Minimum edge required to place bet
        """
        self.initial_bankroll = initial_bankroll
        self.bankroll = initial_bankroll
        self.kelly_fraction = kelly_fraction
        self.min_edge = min_edge
        self.bet_history = []
        
    def calculate_kelly(self, probability, odds):
        """Calculate Kelly Criterion bet size."""
        p = probability
        b = odds - 1
        kelly = (p * b - (1 - p)) / b
        return max(kelly, 0)
    
    def identify_value_bet(self, predictions, odds_dict):
        """
        Identify best value betting opportunity.
        
        Args:
            predictions: Dict with your probabilities {'H': 0.5, 'D': 0.3, 'A': 0.2}
            odds_dict: Dict with bookmaker odds {'H': 2.0, 'D': 3.5, 'A': 4.0}
        
        Returns:
            best_bet: Tuple (outcome, probability, odds, edge, kelly) or None
        """
        best_bet = None
        best_kelly = 0
        
        for outcome in ['H', 'D', 'A']:
            prob = predictions[outcome]
            odds = odds_dict[outcome]
            implied_prob = 1 / odds
            edge = prob - implied_prob
            
            # Only consider if edge exceeds minimum
            if edge > self.min_edge:
                kelly = self.calculate_kelly(prob, odds)
                
                if kelly > best_kelly:
                    best_kelly = kelly
                    best_bet = (outcome, prob, odds, edge, kelly)
        
        return best_bet
    
    def place_bet(self, match_id, predictions, odds_dict, actual_result):
        """
        Place a bet and update bankroll.
        
        Args:
            match_id: Match identifier
            predictions: Your probability predictions
            odds_dict: Bookmaker odds
            actual_result: Actual match result ('H', 'D', or 'A')
        
        Returns:
            bet_record: Dictionary with bet details
        """
        # Identify value bet
        value_bet = self.identify_value_bet(predictions, odds_dict)
        
        if value_bet is None:
            # No value bet found
            return None
        
        outcome, prob, odds, edge, kelly = value_bet
        
        # Calculate bet size
        bet_size = self.bankroll * kelly * self.kelly_fraction
        bet_size = min(bet_size, self.bankroll * 0.1)  # Max 10% of bankroll per bet
        bet_size = max(bet_size, 0)
        
        # Determine profit/loss
        if outcome == actual_result:
            profit = bet_size * (odds - 1)
            win = True
        else:
            profit = -bet_size
            win = False
        
        # Update bankroll
        old_bankroll = self.bankroll
        self.bankroll += profit
        
        # Record bet
        bet_record = {
            'match_id': match_id,
            'prediction': outcome,
            'actual': actual_result,
            'probability': prob,
            'odds': odds,
            'edge': edge,
            'kelly': kelly,
            'bet_size': bet_size,
            'profit': profit,
            'bankroll_before': old_bankroll,
            'bankroll_after': self.bankroll,
            'win': win
        }
        
        self.bet_history.append(bet_record)
        
        return bet_record
    
    def get_performance_metrics(self):
        """
        Calculate comprehensive performance metrics.
        """
        if not self.bet_history:
            return None
        
        df = pd.DataFrame(self.bet_history)
        
        total_bets = len(df)
        wins = df['win'].sum()
        losses = total_bets - wins
        win_rate = wins / total_bets
        
        total_staked = df['bet_size'].sum()
        total_profit = df['profit'].sum()
        roi = (total_profit / total_staked) * 100
        
        avg_win = df[df['win']]['profit'].mean() if wins > 0 else 0
        avg_loss = abs(df[~df['win']]['profit'].mean()) if losses > 0 else 0
        profit_factor = avg_win / avg_loss if avg_loss > 0 else 0
        
        # Calculate max drawdown
        df['cumulative_profit'] = df['profit'].cumsum()
        df['running_max'] = df['cumulative_profit'].cummax()
        df['drawdown'] = df['running_max'] - df['cumulative_profit']
        max_drawdown = df['drawdown'].max()
        
        # Calculate Sharpe ratio (simplified)
        returns = df['profit'] / df['bet_size']
        sharpe = returns.mean() / returns.std() if returns.std() > 0 else 0
        
        return {
            'Total Bets': total_bets,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Total Staked': total_staked,
            'Total Profit': total_profit,
            'ROI (%)': roi,
            'Final Bankroll': self.bankroll,
            'Bankroll Growth (%)': ((self.bankroll - self.initial_bankroll) / self.initial_bankroll) * 100,
            'Avg Win': avg_win,
            'Avg Loss': avg_loss,
            'Profit Factor': profit_factor,
            'Max Drawdown': max_drawdown,
            'Sharpe Ratio': sharpe
        }
    
    def get_bet_history_df(self):
        """Return bet history as DataFrame."""
        return pd.DataFrame(self.bet_history) if self.bet_history else None

print("SoccerBettingSystem class created!")

---

# 2. Generate Realistic Season Data

Create a full season with realistic odds and predictions.

In [ ]:
# Generate Premier League season
np.random.seed(42)

num_matches = 380
season_data = []

for i in range(num_matches):
    # True probabilities (unknown in real betting)
    home_prob_true = np.random.uniform(0.35, 0.55)
    draw_prob_true = np.random.uniform(0.20, 0.30)
    away_prob_true = 1 - home_prob_true - draw_prob_true
    
    # Bookmaker odds (with 5% margin)
    overround = 1.05
    home_odds = (1 / home_prob_true) * overround
    draw_odds = (1 / draw_prob_true) * overround
    away_odds = (1 / away_prob_true) * overround
    
    # Your model's predictions (with some noise/error)
    prediction_error = np.random.normal(0, 0.04)  # 4% std error
    home_prob_pred = np.clip(home_prob_true + prediction_error, 0.1, 0.9)
    draw_prob_pred = np.clip(draw_prob_true + prediction_error, 0.1, 0.9)
    
    # Normalize predictions
    total = home_prob_pred + draw_prob_pred
    if total > 0.95:
        away_prob_pred = max(1 - total, 0.05)
    else:
        away_prob_pred = 1 - home_prob_pred - draw_prob_pred
    
    # Simulate actual result
    outcome_rand = np.random.random()
    if outcome_rand < home_prob_true:
        result = 'H'
    elif outcome_rand < home_prob_true + draw_prob_true:
        result = 'D'
    else:
        result = 'A'
    
    season_data.append({
        'match_id': i + 1,
        'home_team': f'Team {(i % 20) + 1}',
        'away_team': f'Team {((i + 10) % 20) + 1}',
        'home_odds': home_odds,
        'draw_odds': draw_odds,
        'away_odds': away_odds,
        'pred_home': home_prob_pred,
        'pred_draw': draw_prob_pred,
        'pred_away': away_prob_pred,
        'result': result
    })

season_df = pd.DataFrame(season_data)

print(f"Generated {len(season_df)} matches")
print(f"\nSample data:")
season_df.head()

---

# 3. Run Complete Betting System

Backtest the system on the full season.

In [ ]:
# Initialize betting system
betting_system = SoccerBettingSystem(
    initial_bankroll=1000,
    kelly_fraction=0.5,  # Half Kelly for safety
    min_edge=0.03  # Minimum 3% edge required
)

# Run through season
for _, match in season_df.iterrows():
    predictions = {
        'H': match['pred_home'],
        'D': match['pred_draw'],
        'A': match['pred_away']
    }
    
    odds = {
        'H': match['home_odds'],
        'D': match['draw_odds'],
        'A': match['away_odds']
    }
    
    betting_system.place_bet(
        match_id=match['match_id'],
        predictions=predictions,
        odds_dict=odds,
        actual_result=match['result']
    )

# Get performance metrics
metrics = betting_system.get_performance_metrics()

print("\n" + "="*60)
print("BETTING SYSTEM PERFORMANCE REPORT")
print("="*60 + "\n")

for key, value in metrics.items():
    if 'Rate' in key or 'ROI' in key or 'Growth' in key:
        print(f"{key:.<40} {value:>15.2f}%")
    elif 'Ratio' in key or 'Factor' in key:
        print(f"{key:.<40} {value:>15.2f}")
    elif isinstance(value, float):
        print(f"{key:.<40} ${value:>14,.2f}")
    else:
        print(f"{key:.<40} {value:>15,}")

print("\n" + "="*60)

---

# 4. Comprehensive Performance Dashboard

In [ ]:
# Get bet history
bet_history = betting_system.get_bet_history_df()

if bet_history is not None and len(bet_history) > 0:
    # Calculate cumulative metrics
    bet_history['cumulative_profit'] = bet_history['profit'].cumsum()
    bet_history['cumulative_roi'] = (bet_history['cumulative_profit'] / 
                                      bet_history['bet_size'].cumsum()) * 100
    
    # Create comprehensive dashboard
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Bankroll evolution
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(bet_history['match_id'], bet_history['bankroll_after'], 
             linewidth=2, color='steelblue')
    ax1.axhline(betting_system.initial_bankroll, color='red', linestyle='--', 
                alpha=0.5, label='Initial Bankroll')
    ax1.fill_between(bet_history['match_id'], betting_system.initial_bankroll, 
                      bet_history['bankroll_after'], 
                      where=bet_history['bankroll_after'] >= betting_system.initial_bankroll,
                      alpha=0.3, color='green', label='Profit')
    ax1.fill_between(bet_history['match_id'], betting_system.initial_bankroll, 
                      bet_history['bankroll_after'],
                      where=bet_history['bankroll_after'] < betting_system.initial_bankroll,
                      alpha=0.3, color='red', label='Loss')
    ax1.set_xlabel('Match Number', fontsize=11)
    ax1.set_ylabel('Bankroll ($)', fontsize=11)
    ax1.set_title('Bankroll Evolution Over Season', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # 2. Cumulative profit
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(bet_history['match_id'], bet_history['cumulative_profit'], 
             linewidth=2, color='green')
    ax2.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Match Number', fontsize=10)
    ax2.set_ylabel('Cumulative Profit ($)', fontsize=10)
    ax2.set_title('Cumulative Profit', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. ROI evolution
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(bet_history['match_id'], bet_history['cumulative_roi'], 
             linewidth=2, color='purple')
    ax3.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Match Number', fontsize=10)
    ax3.set_ylabel('ROI (%)', fontsize=10)
    ax3.set_title('ROI Evolution', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # 4. Bet size distribution
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.hist(bet_history['bet_size'], bins=30, color='orange', alpha=0.7, edgecolor='black')
    ax4.axvline(bet_history['bet_size'].mean(), color='red', linestyle='--', 
                linewidth=2, label=f"Mean: ${bet_history['bet_size'].mean():.2f}")
    ax4.set_xlabel('Bet Size ($)', fontsize=10)
    ax4.set_ylabel('Frequency', fontsize=10)
    ax4.set_title('Bet Size Distribution', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3, axis='y')
    
    # 5. Win/Loss distribution
    ax5 = fig.add_subplot(gs[2, 0])
    win_loss = bet_history['win'].value_counts()
    colors = ['green' if x else 'red' for x in win_loss.index]
    bars = ax5.bar(['Wins', 'Losses'], win_loss.values, color=colors, alpha=0.7, edgecolor='black')
    for bar, val in zip(bars, win_loss.values):
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                f'{val}\n({val/len(bet_history)*100:.1f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax5.set_ylabel('Count', fontsize=10)
    ax5.set_title('Win/Loss Distribution', fontsize=12, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='y')
    
    # 6. Profit by outcome
    ax6 = fig.add_subplot(gs[2, 1])
    profit_by_outcome = bet_history.groupby('prediction')['profit'].sum()
    colors_outcome = ['green' if p > 0 else 'red' for p in profit_by_outcome.values]
    bars = ax6.bar(profit_by_outcome.index, profit_by_outcome.values, 
                   color=colors_outcome, alpha=0.7, edgecolor='black')
    ax6.axhline(0, color='black', linestyle='-', linewidth=0.5)
    ax6.set_xlabel('Prediction', fontsize=10)
    ax6.set_ylabel('Total Profit ($)', fontsize=10)
    ax6.set_title('Profit by Outcome Type', fontsize=12, fontweight='bold')
    ax6.grid(True, alpha=0.3, axis='y')
    
    # 7. Edge vs Profit scatter
    ax7 = fig.add_subplot(gs[2, 2])
    scatter = ax7.scatter(bet_history['edge'] * 100, bet_history['profit'], 
                         c=bet_history['win'], cmap='RdYlGn', 
                         alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    ax7.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax7.axvline(0, color='black', linestyle='--', alpha=0.5)
    ax7.set_xlabel('Edge (%)', fontsize=10)
    ax7.set_ylabel('Profit ($)', fontsize=10)
    ax7.set_title('Edge vs Profit (Green=Win)', fontsize=12, fontweight='bold')
    ax7.grid(True, alpha=0.3)
    
    plt.suptitle('Complete Betting System Dashboard', fontsize=16, fontweight='bold', y=0.995)
    plt.show()
    
    print("Dashboard generated successfully!")
else:
    print("No bets were placed (no value opportunities found).")

---

# 5. Detailed Bet Analysis

In [ ]:
if bet_history is not None and len(bet_history) > 0:
    print("\nTOP 10 MOST PROFITABLE BETS:\n")
    top_bets = bet_history.nlargest(10, 'profit')[['match_id', 'prediction', 'actual', 
                                                     'odds', 'edge', 'bet_size', 'profit', 'win']]
    print(top_bets.to_string(index=False))
    
    print("\n\nTOP 10 WORST BETS:\n")
    worst_bets = bet_history.nsmallest(10, 'profit')[['match_id', 'prediction', 'actual', 
                                                        'odds', 'edge', 'bet_size', 'profit', 'win']]
    print(worst_bets.to_string(index=False))
    
    print("\n\nPERFORMANCE BY PREDICTION TYPE:\n")
    perf_by_type = bet_history.groupby('prediction').agg({
        'match_id': 'count',
        'win': 'sum',
        'profit': 'sum',
        'bet_size': 'sum'
    })
    perf_by_type.columns = ['Bets', 'Wins', 'Total Profit', 'Total Staked']
    perf_by_type['Win Rate'] = (perf_by_type['Wins'] / perf_by_type['Bets']) * 100
    perf_by_type['ROI (%)'] = (perf_by_type['Total Profit'] / perf_by_type['Total Staked']) * 100
    print(perf_by_type)

## Summary

In this notebook, you learned:

✅ **Complete betting system** - End-to-end implementation

✅ **Value bet identification** - Systematic approach

✅ **Kelly Criterion integration** - Optimal bet sizing

✅ **Comprehensive backtesting** - Rigorous evaluation

✅ **Performance dashboard** - Visual analytics

✅ **Risk management** - Bankroll protection

### Key Takeaways

**1. System components**
- Prediction model
- Odds analysis
- Bet sizing (Kelly)
- Risk management
- Performance tracking

**2. Practical considerations**
- Minimum edge threshold
- Maximum bet size limits
- Fractional Kelly for safety
- Comprehensive logging

**3. Performance metrics**
- ROI (primary metric)
- Win rate
- Profit factor
- Max drawdown
- Sharpe ratio

**4. Real-world challenges**
- Prediction accuracy
- Odds availability
- Bankroll management
- Psychological discipline

## Practice Exercises

1. **Modify parameters**: Test different Kelly fractions and minimum edges

2. **Add features**: Incorporate streak detection or form analysis

3. **Risk limits**: Implement daily/weekly loss limits

4. **Multiple bookmakers**: Compare odds across bookmakers

5. **Live tracking**: Build a real-time betting tracker

6. **Optimization**: Find optimal parameters through grid search

7. **Monte Carlo**: Simulate 1000 seasons to test robustness

8. **Production system**: Add logging, error handling, and alerts